# Getting the Bronze Data From External storage

Get the Product Files from the storage 

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
base_path = 'abfss://products@adlsstdout001tgt.dfs.core.windows.net/'

## Explicit Schema

In [0]:
# =========================================================
# STEP 1 — DEFINE EXPLICIT SCHEMA
# =========================================================
# Enterprise pipelines avoid inferSchema in production
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True)
])

In [0]:
from pyspark.sql import functions as F

from pyspark.sql import functions as F

df_bronze = (
    spark.read.format("parquet")
        .schema(products_schema)
        .option("recursiveFileLookup", "true")
        .load(base_path)
        .withColumn("file_path", F.col("_metadata.file_path"))
        .withColumn("year", F.regexp_extract("file_path", r"/(\d{4})/", 1))
        .withColumn("month", F.regexp_extract("file_path", r"/\d{4}/(\d{2})/", 1))
        .withColumn("day", F.regexp_extract("file_path", r"/\d{4}/\d{2}/(\d{2})/", 1))
)

In [0]:
df_bronze.printSchema()

In [0]:
display(df_bronze)

# Data Cleaning

In [0]:
# =========================================================
# STEP 3 — ADD AUDIT COLUMNS
# =========================================================
# Audit columns are important in enterprise systems
# for lineage and troubleshooting

df_audit = (
    df_bronze
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_system",
            F.lit("olist_ecommerce")
        )
)

# =========================================================
# STEP 4 — REMOVE DUPLICATE PRODUCTS
# =========================================================
# Product dimension should contain one unique product_id

df_deduplicated = (
    df_audit
        .dropDuplicates(["product_id"])
)

# =========================================================
# STEP 5 — STANDARDIZE CATEGORY NAMES
# =========================================================
# Enterprise data usually standardizes text columns
# for reporting consistency

df_standardized = (
    df_deduplicated

        # Remove leading/trailing spaces
        .withColumn(
            "product_category_name",
            F.trim(F.col("product_category_name"))
        )

        # Convert to lowercase
        .withColumn(
            "product_category_name",
            F.lower(F.col("product_category_name"))
        )

        # Replace underscores with spaces
        .withColumn(
            "product_category_name",
            F.regexp_replace(
                "product_category_name",
                "_",
                " "
            )
        )
)

# =========================================================
# STEP 6 — HANDLE NULL CATEGORY VALUES
# =========================================================
# Null business categories can break reporting

df_null_handled = (
    df_standardized
        .withColumn(
            "product_category_name",
            F.coalesce(
                F.col("product_category_name"),
                F.lit("unknown")
            )
        )
)

# =========================================================
# STEP 7 — BUSINESS RULE VALIDATIONS
# =========================================================
# Enterprise pipelines validate impossible values

df_validated = (
    df_null_handled

        # Remove products with invalid weight
        .filter(F.col("product_weight_g") > 0)

        # Remove invalid dimensions
        .filter(F.col("product_length_cm") > 0)
        .filter(F.col("product_height_cm") > 0)
        .filter(F.col("product_width_cm") > 0)
)

# =========================================================
# STEP 8 — CREATE DERIVED BUSINESS METRICS
# =========================================================
# Derived columns improve analytics usability

df_enriched = (

    df_validated

        # Product volume calculation
        .withColumn(
            "product_volume_cm3",
            (
                F.col("product_length_cm")
                * F.col("product_height_cm")
                * F.col("product_width_cm")
            )
        )

        # Weight segmentation
        .withColumn(
            "weight_category",

            F.when(
                F.col("product_weight_g") < 500,
                "Light"
            )

            .when(
                F.col("product_weight_g") < 5000,
                "Medium"
            )

            .otherwise("Heavy")
        )
)

# =========================================================
# STEP 9 — CREATE DATA QUALITY FLAGS
# =========================================================
# Recruiters love seeing DQ engineering concepts

df_quality = (

    df_enriched

        .withColumn(

            "is_valid_product",

            F.when(

                (F.col("product_name_lenght") > 0)
                &
                (F.col("product_description_lenght") > 0)
                &
                (F.col("product_photos_qty") > 0),

                True

            ).otherwise(False)
        )
)

# =========================================================
# STEP 10 — FINAL SILVER DATAFRAME
# =========================================================
# This dataframe is ready for:
# - SCD1 Merge
# - Reporting
# - Gold Layer
# - Power BI

df_silver_products = df_quality

In [0]:
display(df_silver_products)

In [0]:
table_exists = spark.catalog.tableExists(
    "ecom.silver.dim_silver_products"
)
# =========================================================
# STEP 12 — INITIAL LOAD
# =========================================================
# First execution creates Silver dimension table

if not table_exists:

    (
        df_silver_products.write
                .format("delta")
                .mode("overwrite")
                .option(
                    "path",
                    "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/Products"
                )
            .saveAsTable("ecom.silver.dim_silver_products")
    )

# =========================================================
# STEP 13 — INCREMENTAL SCD TYPE 1 MERGE
# =========================================================
# Existing products updated
# New products inserted

else:

    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(
        spark,
        "ecom.silver.dim_silver_products"
    )

    (
        delta_table.alias("target")

        .merge(
            df_silver_products.alias("source"),

            """
            target.product_id = source.product_id
            """
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )